In [1]:
# ============================================
# PROJECT 2: Late Delivery Risk Prediction
# Phase 3: Data Cleaning
# Author: Mohan | Unified Mentor Internship
# ============================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Load raw dataset
df = pd.read_csv('../data/APL_Logistics.csv', encoding='latin-1')

print(f"✅ Raw dataset loaded")
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"💾 Memory: {df.memory_usage().sum() / 1024**2:.2f} MB")

✅ Raw dataset loaded
📊 Shape: 180,519 rows × 40 columns
💾 Memory: 55.09 MB


In [2]:
# Create a working copy — NEVER modify the original df
# This is a professional habit. If something breaks, you can reload from df.

df_clean = df.copy()
print(f"✅ Working copy created: df_clean")
print(f"📊 Shape: {df_clean.shape}")

✅ Working copy created: df_clean
📊 Shape: (180519, 40)


In [3]:
# ===== STEP 1: REMOVE DATA LEAKAGE COLUMNS =====
# These columns contain information only known AFTER delivery happens.
# Using them = cheating = useless model in production.

leakage_columns = [
    'Days for shipping (real)',    # Actual shipping days — unknown at order time
    'Delivery Status',              # Directly states late/on-time — IS the answer
    'Order Status'                  # Filled after order completes
]

print("🚨 Removing data leakage columns:")
for col in leakage_columns:
    print(f"   ❌ Dropping: {col}")

df_clean = df_clean.drop(columns=leakage_columns)

print(f"\n✅ Shape after dropping leakage: {df_clean.shape}")
print(f"   (Removed {len(leakage_columns)} columns)")

🚨 Removing data leakage columns:
   ❌ Dropping: Days for shipping (real)
   ❌ Dropping: Delivery Status
   ❌ Dropping: Order Status

✅ Shape after dropping leakage: (180519, 37)
   (Removed 3 columns)


In [4]:
# ===== STEP 2: REMOVE IDENTIFIER COLUMNS =====
# These uniquely identify customers/orders but have NO predictive power.
# Including them causes overfitting (model memorizes instead of learning).

identifier_columns = [
    'Customer Id',          # Just a number, no business meaning for prediction
    'Customer Fname',       # First name — irrelevant
    'Customer Lname',       # Last name — irrelevant  
    'Customer Email',       # Doesn't exist here, but typically removed
    'Customer Password',    # Doesn't exist here, but always removed
    'Customer Street',      # Too granular (180K+ unique values)
    'Customer Zipcode',     # Granularity issue
    'Order Customer Id',    # Same as Customer Id
]

# Only drop columns that actually exist (safety check)
identifier_columns = [col for col in identifier_columns if col in df_clean.columns]

print("🔒 Removing identifier columns (privacy + no predictive value):")
for col in identifier_columns:
    print(f"   ❌ Dropping: {col}")

df_clean = df_clean.drop(columns=identifier_columns)

print(f"\n✅ Shape after dropping identifiers: {df_clean.shape}")

🔒 Removing identifier columns (privacy + no predictive value):
   ❌ Dropping: Customer Id
   ❌ Dropping: Customer Fname
   ❌ Dropping: Customer Lname
   ❌ Dropping: Customer Street
   ❌ Dropping: Customer Zipcode
   ❌ Dropping: Order Customer Id

✅ Shape after dropping identifiers: (180519, 31)


In [5]:
# ===== STEP 3: REMOVE OVER-GRANULAR LOCATION COLUMNS =====
# We keep Market, Region, Country (high-level)
# We drop City, State, Street (too granular — causes overfitting)

# Check what location columns exist
location_columns_to_drop = [
    'Customer City',       # Too many unique values
    'Customer State',      # State-level, but we have Country
    'Order City',          # Too granular
    'Order State',         # We keep Order Region/Country instead
    'Product Name',        # Has too many unique values
    'Category Id',         # Duplicate of Category Name
    'Department Id',       # Duplicate of Department Name
]

# Only drop columns that actually exist
location_columns_to_drop = [col for col in location_columns_to_drop if col in df_clean.columns]

print("🌍 Removing over-granular columns:")
for col in location_columns_to_drop:
    unique_count = df_clean[col].nunique()
    print(f"   ❌ {col}: {unique_count:,} unique values")

df_clean = df_clean.drop(columns=location_columns_to_drop)

print(f"\n✅ Shape after dropping granular columns: {df_clean.shape}")

🌍 Removing over-granular columns:
   ❌ Customer City: 563 unique values
   ❌ Customer State: 46 unique values
   ❌ Order City: 3,597 unique values
   ❌ Order State: 1,089 unique values
   ❌ Product Name: 118 unique values
   ❌ Category Id: 51 unique values
   ❌ Department Id: 11 unique values

✅ Shape after dropping granular columns: (180519, 24)


In [6]:
# Verify which columns remain
print("="*60)
print(f"REMAINING COLUMNS ({df_clean.shape[1]} total):")
print("="*60)

for i, col in enumerate(df_clean.columns, 1):
    dtype = df_clean[col].dtype
    nunique = df_clean[col].nunique()
    print(f"{i:2}. {col:<35} | {str(dtype):<10} | {nunique:>6,} unique")

REMAINING COLUMNS (24 total):
 1. Type                                | object     |      4 unique
 2. Days for shipment (scheduled)       | int64      |      4 unique
 3. Benefit per order                   | float64    | 21,998 unique
 4. Sales per customer                  | float64    |  2,927 unique
 5. Late_delivery_risk                  | int64      |      2 unique
 6. Category Name                       | object     |     50 unique
 7. Customer Country                    | object     |      2 unique
 8. Customer Segment                    | object     |      3 unique
 9. Department Name                     | object     |     11 unique
10. Latitude                            | float64    | 11,250 unique
11. Longitude                           | float64    |  4,487 unique
12. Market                              | object     |      5 unique
13. Order Country                       | object     |    164 unique
14. Order Item Discount                 | float64    |  1,017 unique
15. 

In [7]:
# ===== STEP 4: HANDLE MISSING VALUES =====
# Check current missing values (most were in dropped columns)

missing_after = df_clean.isnull().sum()
missing_after = missing_after[missing_after > 0]

if len(missing_after) > 0:
    print("Missing values found:")
    print(missing_after)
    
    # Fill any remaining missing values
    # For numeric: fill with median
    # For categorical: fill with mode
    for col in missing_after.index:
        if df_clean[col].dtype in ['int64', 'float64']:
            median_val = df_clean[col].median()
            df_clean[col].fillna(median_val, inplace=True)
            print(f"   ✅ {col}: filled with median ({median_val})")
        else:
            mode_val = df_clean[col].mode()[0]
            df_clean[col].fillna(mode_val, inplace=True)
            print(f"   ✅ {col}: filled with mode ('{mode_val}')")
else:
    print("✅ No missing values remaining!")
    print("   (The 11 missing cells were in columns we already dropped)")

# Final verification
print(f"\nFinal missing values: {df_clean.isnull().sum().sum()}")

✅ No missing values remaining!
   (The 11 missing cells were in columns we already dropped)

Final missing values: 0


In [8]:
# ===== STEP 5: REMOVE DUPLICATE ROWS =====
duplicates_before = df_clean.duplicated().sum()
print(f"Duplicates found: {duplicates_before:,}")

if duplicates_before > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"✅ Duplicates removed")
    print(f"📊 New shape: {df_clean.shape}")
else:
    print("✅ No duplicates — dataset is clean!")

Duplicates found: 0
✅ No duplicates — dataset is clean!


In [9]:
# ===== STEP 6: SANITY CHECK ON NUMERIC COLUMNS =====
# Look for impossible or suspicious values

print("="*60)
print("NUMERIC COLUMN SANITY CHECK")
print("="*60)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    min_val = df_clean[col].min()
    max_val = df_clean[col].max()
    mean_val = df_clean[col].mean()
    
    # Flag suspicious values
    flag = ""
    if min_val < 0 and col not in ['Benefit per order', 'Order Profit Per Order', 
                                     'Order Item Profit Ratio', 'Latitude', 'Longitude']:
        flag = " ⚠️ NEGATIVE"
    
    print(f"{col:<35} | min: {min_val:>12.2f} | max: {max_val:>12.2f} | mean: {mean_val:>10.2f}{flag}")

NUMERIC COLUMN SANITY CHECK
Days for shipment (scheduled)       | min:         0.00 | max:         4.00 | mean:       2.93
Benefit per order                   | min:     -4274.98 | max:       911.80 | mean:      21.97
Sales per customer                  | min:         7.49 | max:      1939.99 | mean:     183.11
Late_delivery_risk                  | min:         0.00 | max:         1.00 | mean:       0.55
Latitude                            | min:       -33.94 | max:        48.78 | mean:      29.72
Longitude                           | min:      -158.03 | max:       115.26 | mean:     -84.92
Order Item Discount                 | min:         0.00 | max:       500.00 | mean:      20.66
Order Item Discount Rate            | min:         0.00 | max:         0.25 | mean:       0.10
Order Item Product Price            | min:         9.99 | max:      1999.99 | mean:     141.23
Order Item Profit Ratio             | min:        -2.75 | max:         0.50 | mean:       0.12
Order Item Quantity   

In [10]:
# ===== STEP 7: CLEAN STRING COLUMNS =====
# Remove leading/trailing whitespace, standardize text

text_cols = df_clean.select_dtypes(include=['object']).columns

print("Cleaning text columns:")
for col in text_cols:
    before = df_clean[col].nunique()
    df_clean[col] = df_clean[col].astype(str).str.strip()
    after = df_clean[col].nunique()
    if before != after:
        print(f"   ✅ {col}: {before} → {after} unique (whitespace removed)")

print("\n✅ All text columns cleaned")

Cleaning text columns:

✅ All text columns cleaned


In [11]:
# ===== PHASE 3 SUMMARY =====
print("="*60)
print("PHASE 3 SUMMARY — Data Cleaning Complete")
print("="*60)

print(f"\nORIGINAL:")
print(f"  Rows: {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")

print(f"\nCLEANED:")
print(f"  Rows: {df_clean.shape[0]:,}")
print(f"  Columns: {df_clean.shape[1]}")
print(f"  Removed: {df.shape[1] - df_clean.shape[1]} columns")

print(f"\nTARGET VARIABLE PRESERVED:")
print(f"  Late_delivery_risk in dataset: {'✅ Yes' if 'Late_delivery_risk' in df_clean.columns else '❌ NO — ERROR!'}")
print(f"  Distribution: {df_clean['Late_delivery_risk'].value_counts().to_dict()}")

print(f"\nDATA QUALITY:")
print(f"  Missing values: {df_clean.isnull().sum().sum()}")
print(f"  Duplicates: {df_clean.duplicated().sum()}")

print(f"\nCOLUMN BREAKDOWN:")
print(f"  Numeric: {df_clean.select_dtypes(include=[np.number]).shape[1]}")
print(f"  Categorical: {df_clean.select_dtypes(include=['object']).shape[1]}")

print(f"\nMEMORY:")
print(f"  Before: {df.memory_usage().sum() / 1024**2:.2f} MB")
print(f"  After:  {df_clean.memory_usage().sum() / 1024**2:.2f} MB")
print(f"  Saved:  {(df.memory_usage().sum() - df_clean.memory_usage().sum()) / 1024**2:.2f} MB")

PHASE 3 SUMMARY — Data Cleaning Complete

ORIGINAL:
  Rows: 180,519
  Columns: 40

CLEANED:
  Rows: 180,519
  Columns: 24
  Removed: 16 columns

TARGET VARIABLE PRESERVED:
  Late_delivery_risk in dataset: ✅ Yes
  Distribution: {1: 98977, 0: 81542}

DATA QUALITY:
  Missing values: 0
  Duplicates: 0

COLUMN BREAKDOWN:
  Numeric: 15
  Categorical: 9

MEMORY:
  Before: 55.09 MB
  After:  33.05 MB
  Saved:  22.04 MB


In [12]:
# ===== SAVE CLEANED DATASET =====
# This is the file we'll use in Phase 4 (EDA) and beyond

output_path = '../data/APL_Logistics_cleaned.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Cleaned dataset saved!")
print(f"📁 Location: {output_path}")
print(f"📊 Shape: {df_clean.shape}")
print(f"\n💡 In future phases, load this with:")
print(f"   df = pd.read_csv('../data/APL_Logistics_cleaned.csv')")

✅ Cleaned dataset saved!
📁 Location: ../data/APL_Logistics_cleaned.csv
📊 Shape: (180519, 24)

💡 In future phases, load this with:
   df = pd.read_csv('../data/APL_Logistics_cleaned.csv')


In [14]:
# Save phase 3 summary (with UTF-8 encoding to support emojis)
summary = f"""
PHASE 3 SUMMARY — Data Cleaning Complete
==========================================
INPUT:  APL_Logistics.csv ({df.shape[0]:,} rows × {df.shape[1]} columns)
OUTPUT: APL_Logistics_cleaned.csv ({df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns)

ACTIONS TAKEN:
1. Removed 3 DATA LEAKAGE columns:
   - Days for shipping (real)
   - Delivery Status
   - Order Status

2. Removed identifier/PII columns (privacy + no predictive value):
   - Customer Fname, Customer Lname
   - Customer Street, Customer Zipcode
   - Customer Id, Order Customer Id

3. Removed over-granular columns (overfitting risk):
   - Customer City, Customer State
   - Order City, Order State
   - Product Name, Category Id, Department Id

4. Cleaned text columns (whitespace, standardization)
5. Verified no missing values, no duplicates

REMAINING FEATURES: {df_clean.shape[1]} columns
TARGET: Late_delivery_risk (preserved)

READY FOR: Phase 4 — Exploratory Data Analysis (EDA)
"""
print(summary)

# THE FIX: add encoding='utf-8' to handle special characters
with open('../phase3_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary)
print("Saved to phase3_summary.txt")


PHASE 3 SUMMARY — Data Cleaning Complete
INPUT:  APL_Logistics.csv (180,519 rows × 40 columns)
OUTPUT: APL_Logistics_cleaned.csv (180,519 rows × 24 columns)

ACTIONS TAKEN:
1. Removed 3 DATA LEAKAGE columns:
   - Days for shipping (real)
   - Delivery Status
   - Order Status

2. Removed identifier/PII columns (privacy + no predictive value):
   - Customer Fname, Customer Lname
   - Customer Street, Customer Zipcode
   - Customer Id, Order Customer Id

3. Removed over-granular columns (overfitting risk):
   - Customer City, Customer State
   - Order City, Order State
   - Product Name, Category Id, Department Id

4. Cleaned text columns (whitespace, standardization)
5. Verified no missing values, no duplicates

REMAINING FEATURES: 24 columns
TARGET: Late_delivery_risk (preserved)

READY FOR: Phase 4 — Exploratory Data Analysis (EDA)

Saved to phase3_summary.txt
